# Modelos COMPAS (réplica del paper)

<!-- Este notebook:
- Carga los CSV generados en el notebook de preprocesamiento
- Entrena los modelos del paper:
  - Logistic Regression
  - SVM
  - XGBoost
- Usa los conjuntos de 2, 7 y 8 features
- Calcula métricas:
  - Accuracy
  - F1
  - False Positive / False Negative por raza
- Guarda todo en un archivo JSON -->


In [14]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from xgboost import XGBClassifier


## 1. Cargar datos preprocesados

In [ ]:
BASE_DIR = Path("data/processed")

df2 = pd.read_csv(BASE_DIR / "compas_features_2.csv") # 2 caracteristicas
df7 = pd.read_csv(BASE_DIR / "compas_features_7.csv") # 7 caracteristicas
df8 = pd.read_csv(BASE_DIR / "compas_preprocessed.csv") # 8 caracteristicas

TARGET = "two_year_recid"

print(df2.shape, df7.shape, df8.shape)


(6142, 3) (6142, 8) (6142, 9)


## 2. Funciones de evaluación

In [9]:
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    }


## 3. Función de entrenamiento

In [10]:
def run_model(model, df, name):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    metrics = compute_metrics(y_test, preds)
    return {"model": name, "metrics": metrics}


## 4. Ejecutar experimentos

In [13]:
results = []

# Logistic Regression
lr = LogisticRegression(max_iter=2000)

results.append(run_model(lr, df2, "LR_2_features"))
results.append(run_model(lr, df7, "LR_7_features"))
results.append(run_model(lr, df8, "LR_8_features"))

# SVM
svm = SVC(kernel="rbf")

results.append(run_model(svm, df2, "SVM_2_features"))
results.append(run_model(svm, df7, "SVM_7_features"))

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1)

results.append(run_model(xgb, df8, "XGB_8_features"))

results


[{'model': 'LR_2_features',
  'metrics': {'accuracy': 0.6696501220504475,
   'precision': np.float64(0.6729166666666667),
   'recall': np.float64(0.5646853146853147),
   'f1_score': 0.6140684410646388,
   'true_negatives': np.int64(500),
   'false_positives': np.int64(157),
   'false_negatives': np.int64(249),
   'true_positives': np.int64(323)}},
 {'model': 'LR_7_features',
  'metrics': {'accuracy': 0.661513425549227,
   'precision': np.float64(0.6535433070866141),
   'recall': np.float64(0.5804195804195804),
   'f1_score': 0.6148148148148148,
   'true_negatives': np.int64(481),
   'false_positives': np.int64(176),
   'false_negatives': np.int64(240),
   'true_positives': np.int64(332)}},
 {'model': 'LR_8_features',
  'metrics': {'accuracy': 0.661513425549227,
   'precision': np.float64(0.6505791505791506),
   'recall': np.float64(0.5891608391608392),
   'f1_score': 0.618348623853211,
   'true_negatives': np.int64(476),
   'false_positives': np.int64(181),
   'false_negatives': np.int

## 5. Guardar resultados

In [8]:
OUTPUT_FILE = BASE_DIR / "results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print("Resultados guardados en:", OUTPUT_FILE)


Resultados guardados en: data\processed\results.json
